## **DS Graph notebook** 

### Imports

In [ ]:
#imports

import csv

import math
import numpy as np
import pandas as pd
import psycopg2
import neo4j
import json
import ast
from IPython.display import display

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

### Set up postgres

In [ ]:
#postgres
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)
cursor = connection.cursor()

In [ ]:
def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)

In [ ]:
def delimit_single_quotes(string_value):
    return string_value.replace("'", "\\'")

In [ ]:
def words_from_list(list_of_words):
    if list_of_words == "Unknown":
        return list_of_words
    text_to_return = ""
    for count, word in enumerate(ast.literal_eval(list_of_words)):
        text_to_return += word
        if count < len(ast.literal_eval(list_of_words)) - 1:
            text_to_return += ", "
    return text_to_return

In [ ]:
def format_text(title, budget, genres, unused_id, keywords, overview, popularity, vote_average, vote_count, 
                cast, director, executive_producer, writer, original_music_composer, director_photography, 
                production_country, production_company, languages):
    return f"{title} (Genres: {genres}. Keywords: {keywords}): {overview} {title} is popular among {popularity} percent of viewers. {vote_count} viewers gave this movie an average rating of {vote_average} out of 10. The cast includes {cast}. {title} was directed by {director} with {words_from_list(executive_producer)} as the executive producer(s). The movie was written by {words_from_list(writer)}. The music was originally composed by {words_from_list(original_music_composer)}, and the photography was directed by {words_from_list(director_photography)}. {title} was produced with a budget of {budget} by {words_from_list(production_company)} in {words_from_list(production_country)}."

In [ ]:
def get_embeddings(text):
    model = SentenceTransformer("all-MiniLM-L6-v2") # Source: https://sbert.net/docs/sentence_transformer/usage/usage.html
    embeddings = model.encode(text) # Result in a 1D array with 384 elements.
    embeddings_array = np.array(embeddings)
    print("Shape of embeddings:", embeddings_array.shape)
    return embeddings_array

### Set up clean movies table

In [ ]:
connection.rollback()

query = """

drop table if exists cleanmovies

"""

cursor.execute(query)

connection.commit()

In [ ]:
connection.rollback()

query = """

create table cleanmovies (
    index numeric,
    title text,
    budget numeric,
    genres text,
    id numeric,
    keywords text,
    overview text,
    popularity numeric,
    vote_average numeric,
    vote_count numeric,
    cast_ text,
    director text,
    executive_producer text,
    writer text,
    original_music_composer text,
    director_photography text,
    production_country text,
    production_company varchar,
    languages text  
)

"""

cursor.execute(query)

connection.commit()

In [ ]:
connection.rollback()

query = """

copy cleanmovies
from '/user/projects/project-3-richardshelby/data/interim/cleanmovies.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from cleanmovies

"""
cleanmovies = my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [ ]:
cleanmovies

In [ ]:
print('Minimum rating:' + ' ' + str(np.min(cleanmovies['vote_average'])))
print('Maximum rating:' + ' ' + str(np.max(cleanmovies['vote_average'])))
print('Minimum popularity:' + ' ' +  str(np.min(cleanmovies['popularity'])))
print('Maximum popularity:' + ' ' +  str(np.max(cleanmovies['popularity'])))

In [ ]:
vector_embeddings = pd.read_csv("../data/interim/Summarized_Text_3.0_Vector_Embeddings.csv")

In [ ]:
vector_embeddings_clean = np.nan_to_num(vector_embeddings.values)
similarities = cosine_similarity(vector_embeddings_clean, vector_embeddings_clean)
pd.DataFrame(similarities)

In [ ]:
cleanmovies = cleanmovies.fillna('Unknown')

In [ ]:
cleanmovies['title'][123]

### Neo4j set up (using code from 1.0-rjs-eda and 3.0-ah-embeddings-eda)

In [ ]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("***","***"))

In [ ]:
session = driver.session(database="neo4j")

In [ ]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [ ]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [ ]:
def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        match (n) 
        return n.title as movie_title, labels(n) as labels
        order by n.title
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.title as movie_title_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.title as movie_title_2, labels(n2) as node_2_labels
        order by movie_title_1, movie_title_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.1f}')
    print("-------------------------")

In [ ]:
def my_neo4j_create_relationship_two_way(from_movie_id, to_movie_id, weight):
    "create relationships two way between two movies with a weight"
    
    query = """
    
    MATCH (from:Movie), 
          (to:Movie)
    WHERE from.id = $from_movie_id and to.id = $to_movie_id
    MERGE (from)-[r1:SIMILAR]->(to)
    MERGE (to)-[r2:SIMILAR]->(from)
    SET r1.weight = $weight,
        r2.weight = $weight
    
    """
    
    session.run(query, from_movie_id=from_movie_id, to_movie_id=to_movie_id, weight=weight)

In [ ]:
def my_neo4j_create_movie_nodes(movies, movie_count):
    """
        movies: pandas dataframe of movies
        movie_count: the number of movies to create nodes for
    """

    nquery = "CREATE \n"
    first = True

    for _, movie in movies.head(movie_count).iterrows():
        if first:
            first = False
        else:
            nquery += ",\n"
            nquery += "(:Movie {"
            nquery += f"title: '{delimit_single_quotes(movie['title'])}', "
            nquery += f"id: {movie['id']}, "
            nquery += f"budget: {movie['budget']}, "
            # this isn't completely accurate. the kaggle set shows new lines but the CSV does not 
            #    have this information. so we have no way to differentiate a movie listed with a 
            #    single genre like "Action Adventure" from one with two genres "Action" and "Adventure"
            #    probably fine, but may be worth mentioning
            nquery += f"genres: '{delimit_single_quotes(movie['genres'])}', "
            nquery += f"keywords: '{delimit_single_quotes(movie['keywords'])}', "
            nquery += f"overview: '{delimit_single_quotes(movie['overview'])}', "
            nquery += f"popularity: {movie['popularity']}, "
            nquery += f"vote_average: {movie['vote_average']}, "
            nquery += f"vote_count: {movie['vote_count']}, "

            # same problem, no delimiter between people. 
            #    space used to seperate first name, last name, and people
            nquery += f"movie_cast: '{delimit_single_quotes(movie['cast_'])}', "

            nquery += f"director: '{delimit_single_quotes(movie['director'])}', "
            nquery += f"director: '{delimit_single_quotes(movie['director'])}', "
            nquery += f"production_company: '{delimit_single_quotes(movie['production_company'])}', "
            
            nquery += "})"

    nquery += ";"
    session.run(nquery)

In [ ]:
def my_neo4j_create_movie_nodes(movies, movie_count): 
    """
        movies: pandas dataframe of movies
        movie_count: the number of movies to create nodes for
    """

    nquery = "CREATE \n"
    first = True

    for _, movie in movies.head(movie_count).iterrows():
        if first:
            first = False
        else:
            nquery += ",\n"
        nquery += "(:Movie {"
        nquery += f"title: '{delimit_single_quotes(movie['title'])}', "
        nquery += f"budget: {movie['budget']}, "
        nquery += f"genres: '{delimit_single_quotes(movie['genres'])}', "
        nquery += f"id: {movie['id']}, "
        nquery += f"keywords: '{delimit_single_quotes(movie['keywords'])}', "
        nquery += f"overview: '{delimit_single_quotes(movie['overview'])}', "
        nquery += f"popularity: {movie['popularity']}, "
        nquery += f"vote_average: {movie['vote_average']}, "
        nquery += f"vote_count: {movie['vote_count']}, "
        nquery += f"movie_cast: '{delimit_single_quotes(movie['cast_'])}', "
        nquery += f"director: '{delimit_single_quotes(movie['director'])}', "
        nquery += f"executive_producer: '{delimit_single_quotes(words_from_list(movie['executive_producer']))}', "
        nquery += f"writer: '{delimit_single_quotes(words_from_list(movie['writer']))}', "
        nquery += f"original_music_composer: '{delimit_single_quotes(words_from_list(movie['original_music_composer']))}', "
        nquery += f"director_photography: '{delimit_single_quotes(words_from_list(movie['director_photography']))}', "
        nquery += f"production_country: '{delimit_single_quotes(words_from_list(movie['production_country']))}', "
        nquery += f"production_company: '{delimit_single_quotes(words_from_list(movie['production_company']))}'"
        nquery += "})"
    nquery += ";"
    session.run(nquery)

In [ ]:
def my_neo4j_create_top_3_similarity_links(movies, movie_count, cosine_similarity_matrix):
    """
        movies: pandas dataframe of movies
        movie_count: the number of movies to create nodes for        
        cosine_similarity_matrix: similarity matrix in the range [-1, 1] where 
            -1 is opposite taste
             0 is no similarity
             1 is perfect match
             
        note - this is hardcoded to top 3 but we can make it general if needed
    """

    for movie_idx in range(movie_count):
        # track top 3 movie by index and score
        similar_1_idx = -1
        similar_2_idx = -1
        similar_3_idx = -1
        similar_1_score = -1
        similar_2_score = -1
        similar_3_score = -1

        for movie_idx_compare in range(movie_count):
            
            # only compare movies to other movies
            if movie_idx == movie_idx_compare:
                continue
                
            similarity_score = cosine_similarity_matrix[movie_idx][movie_idx_compare]

            if similarity_score > similar_1_score:
                similar_3_idx = similar_2_idx
                similar_3_score = similar_2_score

                similar_2_idx = similar_1_idx            
                similar_2_score = similar_1_score

                similar_1_idx = movie_idx_compare            
                similar_1_score = similarity_score

            elif similarity_score > similar_2_score:
                similar_3_idx = similar_2_idx
                similar_3_score = similar_2_score

                similar_2_idx = movie_idx_compare
                similar_2_score = similarity_score

            elif similarity_score > similar_3_score:
                similar_3_idx = movie_idx_compare
                similar_3_score = similarity_score

        most_similar_array = [
            (similar_1_idx, similar_1_score),
            (similar_2_idx, similar_2_score),
            (similar_3_idx, similar_3_score)
        ]
        
        for similar_idx, similar_score in most_similar_array:
            my_neo4j_create_relationship_two_way(
                from_movie_id = movies.iloc[movie_idx].id, 
                to_movie_id = movies.iloc[similar_idx].id, 
                weight = round(similar_score, 5)
            )

### Set up graph 

In [ ]:
my_neo4j_wipe_out_database()

In [ ]:
my_neo4j_create_movie_nodes(cleanmovies,len(cleanmovies))

In [ ]:
my_neo4j_create_top_3_similarity_links(cleanmovies, len(cleanmovies), similarities)

In [ ]:
my_neo4j_nodes_relationships()

# Page Rank code 

In [ ]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

In [ ]:
query = "CALL gds.graph.project('ds_graph', 'Movie', 'SIMILAR', {relationshipProperties: 'weight'})"
session.run(query)

In [ ]:
query = """

CALL gds.pageRank.stream('ds_graph',
                         { maxIterations: $max_iterations,
                           dampingFactor: $damping_factor
                           }
                         )
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).title AS title, 
       gds.util.asNode(nodeId).popularity AS movie_popularity, 
       gds.util.asNode(nodeId).vote_average AS movie_rating, 
       score as page_rank_score
ORDER BY page_rank_score DESC, movie_popularity ASC, movie_rating DESC

"""

max_iterations = 20
damping_factor = 0.5

influence_rank = my_neo4j_run_query_pandas(query, max_iterations=max_iterations, damping_factor=damping_factor)
influence_rank

In [ ]:
query = """
MATCH (Movie:Movie {title: $source})
CALL gds.pageRank.stream('ds_graph',
                         { maxIterations: $max_iterations,
                           dampingFactor: $damping_factor,
                           sourceNodes: [Movie]
                           }
                         )
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).title AS title, 
       gds.util.asNode(nodeId).popularity AS movie_popularity, 
       gds.util.asNode(nodeId).vote_average AS movie_rating, 
       score as page_rank_score
ORDER BY page_rank_score DESC, movie_popularity ASC, movie_rating DESC 

"""

source = cleanmovies['title'][np.random.randint(0,3224)]
max_iterations = 20
damping_factor = 0.5

print(source)

personalized_influence_rank = my_neo4j_run_query_pandas(query, max_iterations=max_iterations, damping_factor=damping_factor, source=source)
personalized_influence_rank